# 🤖 Step 3: Model Training (Random Forest)

Train a Random Forest classifier using GPS-labeled ground truth points.

## What This Notebook Does:
- ✅ Auto-generate Class 5 (Landmass) training points from high-NDVI areas
- ✅ Merge with existing ground truth CSV
- ✅ Extract 7-band spectral features at GPS locations
- ✅ Train Random Forest Classifier (100 trees)
- ✅ Evaluate performance (accuracy, confusion matrix, feature importance)
- ✅ Save trained model and scaler

## Training Data:
- **Source**: GPS-labeled points from field surveys (`Ground_TruthFinal.csv`)
- **Auto-augmentation**: ~100 Landmass points generated automatically
- **Classes**: 5 (Seagrass=1, Sand=2, Seaweed=3, Water=4, Landmass=5)
- **Features per point**: 7 (B02, B03, B04, B08, NDVI, NDWI, Texture)

## Model Details:
- **Algorithm**: Random Forest (ensemble of decision trees)
- **n_estimators**: 100 trees
- **max_depth**: 15 levels
- **Training time**: ~5-10 seconds on CPU

## Output:
- `../models/coastal_classifier_model.pkl` - Trained model
- `../models/feature_scaler.pkl` - Standardization scaler
- `../models/model_metadata.json` - Accuracy, classes, feature names

---

**Previous:** [02_preprocessing.ipynb](02_preprocessing.ipynb)  
**Next:** [04_prediction_analysis.ipynb](04_prediction_analysis.ipynb)

In [8]:
# ============================================================
# AUTO-GENERATE CLASS 5 (Landmass) TRAINING POINTS
# Add this cell to 02_preprocessing.ipynb or before training
# ============================================================

import rasterio
import numpy as np
import pandas as pd
from rasterio.warp import transform as rio_transform

print("🌿 Generating Class 5 (Landmass/Terrestrial Vegetation) training points...")

# Load the processed image
with rasterio.open("../data/processed/processed_image_with_indices.tif") as src:
    image_data = src.read()
    profile = src.profile
    transform = src.transform
    crs = src.crs

# Extract the index bands
# Band layout: [B02, B03, B04, B08, NDVI, NDWI, Texture] → indices 0-6
ndvi = image_data[4]
ndwi = image_data[5]

# ---------------------------------------------------------------
# LANDMASS CRITERIA:
#   - High NDVI  (> 0.35) → Dense vegetation, definitely not ocean
#   - Low  NDWI  (< -0.1) → Not water, not coastal wetland
#   - Both conditions together = terrestrial landmass
# ---------------------------------------------------------------
landmass_mask = (ndvi > 0.35) & (ndwi < -0.1)

print(f"   Candidate landmass pixels found: {np.sum(landmass_mask):,}")

# Get pixel coordinates of all candidates
rows, cols = np.where(landmass_mask)

# ---------------------------------------------------------------
# SMART SAMPLING: Don't just take random pixels.
# Sample from a spatial grid to ensure geographic spread.
# ---------------------------------------------------------------
N_POINTS = 100  # Adjust: 50 minimum, 150 maximum recommended

if len(rows) < N_POINTS:
    print(f"⚠️  Only {len(rows)} candidate pixels found. Using all of them.")
    sampled_rows, sampled_cols = rows, cols
else:
    # Spatially-distributed sampling using a grid approach
    # Divide the image into a grid and sample uniformly from each cell
    n_grid = int(np.ceil(np.sqrt(N_POINTS)))
    h, w = ndvi.shape
    grid_h = h // n_grid
    grid_w = w // n_grid
    
    sampled_rows, sampled_cols = [], []
    
    for gi in range(n_grid):
        for gj in range(n_grid):
            # Find landmass pixels within this grid cell
            r_min, r_max = gi * grid_h, (gi + 1) * grid_h
            c_min, c_max = gj * grid_w, (gj + 1) * grid_w
            
            in_cell = (rows >= r_min) & (rows < r_max) & \
                      (cols >= c_min) & (cols < c_max)
            
            cell_rows = rows[in_cell]
            cell_cols = cols[in_cell]
            
            if len(cell_rows) > 0:
                # Pick 1 random pixel from this grid cell
                idx = np.random.randint(len(cell_rows))
                sampled_rows.append(cell_rows[idx])
                sampled_cols.append(cell_cols[idx])
    
    sampled_rows = np.array(sampled_rows)
    sampled_cols = np.array(sampled_cols)
    
    print(f"   Spatially sampled: {len(sampled_rows)} points across {n_grid}×{n_grid} grid")

# ---------------------------------------------------------------
# CONVERT pixel (row, col) → geographic (Lon, Lat)
# ---------------------------------------------------------------
# rasterio xy() returns coordinates in the image's CRS
xs, ys = rasterio.transform.xy(transform, sampled_rows, sampled_cols)

# Convert from image CRS → WGS84 (EPSG:4326) to match your CSV format
lons, lats = rio_transform(crs, 'EPSG:4326', xs, ys)

# ---------------------------------------------------------------
# BUILD THE NEW TRAINING ROWS
# ---------------------------------------------------------------
new_points_df = pd.DataFrame({
    'Longitude': lons,
    'Latitude':  lats,
    'Value':     5  # Class 5 = Landmass/Terrestrial Vegetation
})

print(f"\n✅ Generated {len(new_points_df)} Class 5 training points")
print(f"   Lon range: {min(lons):.4f} → {max(lons):.4f}")
print(f"   Lat range: {min(lats):.4f} → {max(lats):.4f}")

# ---------------------------------------------------------------
# MERGE WITH YOUR EXISTING GROUND TRUTH CSV
# ---------------------------------------------------------------
existing_csv_path = "../data/training/trainingdata/Ground_TruthFinal.csv"
existing_df = pd.read_csv(existing_csv_path)

print(f"\n📋 Existing training data: {len(existing_df)} points")
print(f"   Classes before: {sorted(existing_df['Value'].unique())}")

# Append the new class 5 points
combined_df = pd.concat([existing_df, new_points_df], ignore_index=True)

print(f"\n📋 Combined training data: {len(combined_df)} points")
print(f"   Classes after:  {sorted(combined_df['Value'].unique())}")
print(f"   Class distribution:")
for cls in sorted(combined_df['Value'].unique()):
    count = (combined_df['Value'] == cls).sum()
    names = {1:"Seagrass", 2:"Sand", 3:"Seaweed", 4:"Water", 5:"Landmass"}
    print(f"      Class {cls} ({names.get(cls,'Unknown')}): {count} samples")

# Save augmented CSV (keeps original safe)
augmented_path = "../data/training/trainingdata/Ground_TruthFinal_with_landmass.csv"
combined_df.to_csv(augmented_path, index=False)
print(f"\n💾 Saved to: {augmented_path}")
print("   ℹ️  Original CSV untouched. Use augmented file in notebook 03.")

🌿 Generating Class 5 (Landmass/Terrestrial Vegetation) training points...
   Candidate landmass pixels found: 38,570
   Spatially sampled: 33 points across 10×10 grid

✅ Generated 33 Class 5 training points
   Lon range: 120.6030 → 120.6278
   Lat range: 13.9512 → 13.9764

📋 Existing training data: 1056 points
   Classes before: [np.int64(1), np.int64(2), np.int64(3), np.int64(4)]

📋 Combined training data: 1089 points
   Classes after:  [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]
   Class distribution:
      Class 1 (Seagrass): 383 samples
      Class 2 (Sand): 246 samples
      Class 3 (Seaweed): 357 samples
      Class 4 (Water): 70 samples
      Class 5 (Landmass): 33 samples

💾 Saved to: ../data/training/trainingdata/Ground_TruthFinal_with_landmass.csv
   ℹ️  Original CSV untouched. Use augmented file in notebook 03.


## Load Libraries and Data

In [9]:
import pandas as pd
import numpy as np
import rasterio
from rasterio.warp import transform
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
import joblib

# 1. Paths
image_path = "../data/processed/processed_image_with_indices.tif"
# Change 1: Point to the new CSV
csv_path = "../data/training/trainingdata/Ground_TruthFinal_with_landmass.csv"

# Change 2: Include class 5
INCLUDED_CLASSES = [1, 2, 3, 4, 5]

# 2. Load CSV
df = pd.read_csv(csv_path)

# Remove rows with NaN values in the 'Value' column
df = df.dropna(subset=['Value'])

print(f"📍 Original CSV classes: {sorted(df['Value'].unique())}")

# ⭐ STRATEGY: Keep coastal + add vegetation/landmass class for training
# Include vegetation classes if they exist (common codes: 5, 6, 7, etc.)
# Adjust INCLUDED_CLASSES based on your actual ground truth labels
INCLUDED_CLASSES = [1, 2, 3, 4, 5]  # Add landmass/veg class codes here
df = df[df['Value'].isin(INCLUDED_CLASSES)]

print(f"📍 Loaded CSV with {len(df)} valid labels")
print(f"   Classes included: {sorted(df['Value'].unique())}")
print(f"   Class distribution:")
for cls in sorted(df['Value'].unique()):
    count = (df['Value'] == cls).sum()
    class_name = {
        1: "Seagrass",
        2: "Sand",
        3: "Seaweed",
        4: "Water",
        5: "Landmass", 
    }.get(cls, f"Class {cls}")
    print(f"      Class {cls} ({class_name}): {count} samples")

# 3. Extract training data from image at ground truth locations
print("\n📍 Extracting training data from satellite image...")

X = []
y = []

with rasterio.open(image_path) as src:
    img_data = src.read()
    img_crs = src.crs
    
    for i, row in df.iterrows():
        # Get raw coords from CSV
        lon, lat = row['Longitude'], row['Latitude']
        
        # Convert coordinates to the image's coordinate system
        xs, ys = transform('EPSG:4326', img_crs, [lon], [lat])
        x_img, y_img = xs[0], ys[0]
        
        # Get pixel index
        r, c = src.index(x_img, y_img)
        
        # Check if pixel is valid and inside the image
        if 0 <= r < src.height and 0 <= c < src.width:
            val = img_data[:, r, c]
            # Only use pixels without NaN values
            if not np.isnan(val).any():
                X.append(val)
                y.append(int(row['Value']))

X, y = np.array(X), np.array(y)
print(f"\n✅ Extraction Complete! Found {len(X)} valid training points.")
print(f"   Features per point: {X.shape[1]}")
print(f"   Final class distribution:")
for cls in np.unique(y):
    count = np.sum(y == cls)
    percentage = (count / len(y)) * 100
    print(f"      Class {cls}: {count} samples ({percentage:.1f}%)")

📍 Original CSV classes: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]
📍 Loaded CSV with 1089 valid labels
   Classes included: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]
   Class distribution:
      Class 1 (Seagrass): 383 samples
      Class 2 (Sand): 246 samples
      Class 3 (Seaweed): 357 samples
      Class 4 (Water): 70 samples
      Class 5 (Landmass): 33 samples

📍 Extracting training data from satellite image...

✅ Extraction Complete! Found 1089 valid training points.
   Features per point: 7
   Final class distribution:
      Class 1: 383 samples (35.2%)
      Class 2: 246 samples (22.6%)
      Class 3: 357 samples (32.8%)
      Class 4: 70 samples (6.4%)
      Class 5: 33 samples (3.0%)


## Preparing Data for RF

In [10]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Split data into training and testing sets (80-20 split)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"📊 Data Split:")
print(f"  Training samples: {len(X_train)}")
print(f"  Testing samples: {len(X_test)}")
print(f"  Feature dimensions: {X_train.shape[1]}")

# Standardize features (important for many ML algorithms)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
rf_scaler = scaler  # alias so improvement cell can reference it
print("✅ rf_scaler set")

print(f"✅ Data prepared and scaled!")

📊 Data Split:
  Training samples: 871
  Testing samples: 218
  Feature dimensions: 7
✅ rf_scaler set
✅ Data prepared and scaled!


## Train the Model

In [11]:
# ============================================================
# CELL 7 (UPDATED): Train RF + All Improvements, Pick Best
# ============================================================
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, f1_score
from scipy.ndimage import uniform_filter

results = []

# ── Attempt 1: Balanced weights ──
print("📌 Attempt 1: Balanced Class Weights")
rf1 = RandomForestClassifier(n_estimators=100, max_depth=15,
      min_samples_split=5, min_samples_leaf=2,
      class_weight='balanced', random_state=42, n_jobs=-1)
rf1.fit(X_train_scaled, y_train)
pred = rf1.predict(X_test_scaled)
acc  = accuracy_score(y_test, pred)
f1   = f1_score(y_test, pred, average='weighted', zero_division=0)
print(f"   Accuracy: {acc*100:.2f}% | F1: {f1:.4f}")
results.append(("RF + Balanced Weights", acc, f1, rf1, pred))

# ── Attempt 2: Deep trees ──
print("\n📌 Attempt 2: More Trees + No Depth Limit")
rf2 = RandomForestClassifier(n_estimators=300, max_depth=None,
      min_samples_split=3, min_samples_leaf=1,
      class_weight='balanced', random_state=42, n_jobs=-1)
rf2.fit(X_train_scaled, y_train)
pred = rf2.predict(X_test_scaled)
acc  = accuracy_score(y_test, pred)
f1   = f1_score(y_test, pred, average='weighted', zero_division=0)
print(f"   Accuracy: {acc*100:.2f}% | F1: {f1:.4f}")
results.append(("RF + Deep + 300 Trees", acc, f1, rf2, pred))

# ── Attempt 3: Spatial neighborhood features ──
print("\n📌 Attempt 3: Spatial Neighborhood Features")
X_spatial, y_spatial = [], []
with rasterio.open(image_path) as src:
    img_data = src.read()
    img_crs  = src.crs
    h, w     = src.height, src.width
    for i, row in df.iterrows():
        lon, lat = row['Longitude'], row['Latitude']
        xs, ys = transform('EPSG:4326', img_crs, [lon], [lat])
        r, c = src.index(xs[0], ys[0])
        if 1 <= r < h-1 and 1 <= c < w-1:
            neighborhood = img_data[:, r-1:r+2, c-1:c+2]
            if not np.isnan(neighborhood).any():
                center     = img_data[:, r, c]
                neigh_mean = neighborhood.mean(axis=(1,2))
                neigh_std  = neighborhood.std(axis=(1,2))
                X_spatial.append(np.concatenate([center, neigh_mean, neigh_std]))
                y_spatial.append(int(row['Value']))

X_spatial = np.array(X_spatial)
y_spatial  = np.array(y_spatial)
print(f"   Spatial features per point: {X_spatial.shape[1]} (was 7)")

from sklearn.preprocessing import StandardScaler as SS
X_tr_sp, X_te_sp, y_tr_sp, y_te_sp = train_test_split(
    X_spatial, y_spatial, test_size=0.2, random_state=42, stratify=y_spatial)
scaler_sp  = SS()
X_tr_sp_sc = scaler_sp.fit_transform(X_tr_sp)
X_te_sp_sc = scaler_sp.transform(X_te_sp)

rf3 = RandomForestClassifier(n_estimators=300, max_depth=None,
      class_weight='balanced', random_state=42, n_jobs=-1)
rf3.fit(X_tr_sp_sc, y_tr_sp)
pred = rf3.predict(X_te_sp_sc)
acc  = accuracy_score(y_te_sp, pred)
f1   = f1_score(y_te_sp, pred, average='weighted', zero_division=0)
print(f"   Accuracy: {acc*100:.2f}% | F1: {f1:.4f}")
results.append(("RF + Spatial Features", acc, f1, rf3, pred))

# ── Attempt 4: Grid search ──
print("\n📌 Attempt 4: Grid Search Hyperparameter Tuning")
param_grid = {
    'n_estimators': [200, 300], 'max_depth': [None, 20],
    'min_samples_leaf': [1, 2], 'max_features': ['sqrt', 'log2']
}
gs = GridSearchCV(
    RandomForestClassifier(class_weight='balanced', random_state=42, n_jobs=-1),
    param_grid, cv=3, scoring='f1_weighted', n_jobs=-1, verbose=1)
gs.fit(X_train_scaled, y_train)
pred = gs.best_estimator_.predict(X_test_scaled)
acc  = accuracy_score(y_test, pred)
f1   = f1_score(y_test, pred, average='weighted', zero_division=0)
print(f"   Best params: {gs.best_params_}")
print(f"   Accuracy: {acc*100:.2f}% | F1: {f1:.4f}")
results.append(("RF + Grid Search", acc, f1, gs.best_estimator_, pred))

# ── Pick best ──
best_result = max(results, key=lambda x: x[1])
best_name, best_acc, best_f1, best_model, best_pred = best_result

print(f"\n{'='*55}")
print(f"🏆 Best RF: {best_name} — {best_acc*100:.2f}%")
print(f"{'='*55}")

📌 Attempt 1: Balanced Class Weights
   Accuracy: 86.24% | F1: 0.8620

📌 Attempt 2: More Trees + No Depth Limit
   Accuracy: 87.16% | F1: 0.8692

📌 Attempt 3: Spatial Neighborhood Features
   Spatial features per point: 21 (was 7)
   Accuracy: 93.12% | F1: 0.9312

📌 Attempt 4: Grid Search Hyperparameter Tuning
Fitting 3 folds for each of 16 candidates, totalling 48 fits
   Best params: {'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'n_estimators': 200}
   Accuracy: 88.07% | F1: 0.8783

🏆 Best RF: RF + Spatial Features — 93.12%


## Evaluate the Performance


In [12]:
# ============================================================
# CELL 9 (UPDATED): Evaluate Best RF Performance
# ============================================================
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

# Use the correct test set depending on whether spatial model won
if "Spatial" in best_name:
    y_eval_test = y_te_sp
    y_eval_pred = best_pred
else:
    y_eval_test = y_test
    y_eval_pred = best_pred

train_set   = X_tr_sp_sc if "Spatial" in best_name else X_train_scaled
train_label = y_tr_sp    if "Spatial" in best_name else y_train
y_train_pred = best_model.predict(train_set)

train_accuracy = accuracy_score(train_label, y_train_pred)
test_accuracy  = accuracy_score(y_eval_test, y_eval_pred)
precision = precision_score(y_eval_test, y_eval_pred, average='weighted', zero_division=0)
recall    = recall_score(y_eval_test, y_eval_pred, average='weighted', zero_division=0)
f1        = f1_score(y_eval_test, y_eval_pred, average='weighted', zero_division=0)

print("=" * 60)
print(f"📊 BEST RF MODEL: {best_name}")
print("=" * 60)
print(f"\n🎯 Accuracy:")
print(f"   Training: {train_accuracy:.4f} ({train_accuracy*100:.2f}%)")
print(f"   Testing:  {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")
print(f"\n📈 Testing Metrics:")
print(f"   Precision: {precision:.4f}")
print(f"   Recall:    {recall:.4f}")
print(f"   F1-Score:  {f1:.4f}")
print(f"\n📋 Confusion Matrix:")
print(confusion_matrix(y_eval_test, y_eval_pred))
print(f"\n📝 Classification Report:")
print(classification_report(y_eval_test, y_eval_pred, zero_division=0,
      target_names=["Seagrass","Sand","Seaweed","Water","Landmass"]))

# Feature importance — only meaningful for non-spatial model
# For spatial, show top features from the 21-feature set
feature_names = (
    ['B02_c','B03_c','B04_c','B08_c','NDVI_c','NDWI_c','Tex_c',
     'B02_mn','B03_mn','B04_mn','B08_mn','NDVI_mn','NDWI_mn','Tex_mn',
     'B02_sd','B03_sd','B04_sd','B08_sd','NDVI_sd','NDWI_sd','Tex_sd']
    if "Spatial" in best_name else
    ['B02','B03','B04','B08','NDVI','NDWI','Texture']
)
importance = best_model.feature_importances_
top3 = np.argsort(importance)[-3:][::-1]
print(f"\n🔍 Top 3 Most Important Features:")
for i in top3:
    print(f"   {feature_names[i]}: {importance[i]:.4f}")

📊 BEST RF MODEL: RF + Spatial Features

🎯 Accuracy:
   Training: 0.9989 (99.89%)
   Testing:  0.9312 (93.12%)

📈 Testing Metrics:
   Precision: 0.9314
   Recall:    0.9312
   F1-Score:  0.9312

📋 Confusion Matrix:
[[74  1  2  0  0]
 [ 2 44  1  0  2]
 [ 3  2 66  0  0]
 [ 0  0  0 14  0]
 [ 0  2  0  0  5]]

📝 Classification Report:
              precision    recall  f1-score   support

    Seagrass       0.94      0.96      0.95        77
        Sand       0.90      0.90      0.90        49
     Seaweed       0.96      0.93      0.94        71
       Water       1.00      1.00      1.00        14
    Landmass       0.71      0.71      0.71         7

    accuracy                           0.93       218
   macro avg       0.90      0.90      0.90       218
weighted avg       0.93      0.93      0.93       218


🔍 Top 3 Most Important Features:
   B04_mn: 0.0967
   NDVI_c: 0.0845
   B03_mn: 0.0703


## Save the Model

In [13]:
# ============================================================
# CELL 11 (UPDATED): Save Best RF Model
# ============================================================
import os, json

os.makedirs('../models', exist_ok=True)
os.makedirs('../outputs', exist_ok=True)

# Safety check
if 'best_model' not in dir():
    raise RuntimeError("❌ best_model not defined — run Cell 7 first")

# Determine if best model used spatial features
is_spatial    = "Spatial" in best_name
active_scaler = scaler_sp if is_spatial else scaler
n_features    = 21 if is_spatial else 7
feature_names = (
    ['B02_center','B03_center','B04_center','B08_center',
     'NDVI_center','NDWI_center','Texture_center',
     'B02_mean','B03_mean','B04_mean','B08_mean',
     'NDVI_mean','NDWI_mean','Texture_mean',
     'B02_std','B03_std','B04_std','B08_std',
     'NDVI_std','NDWI_std','Texture_std']
    if is_spatial else
    ['B02 (Blue)','B03 (Green)','B04 (Red)','B08 (NIR)','NDVI','NDWI','Texture']
)

joblib.dump(best_model,    '../models/coastal_classifier_model.pkl')
joblib.dump(active_scaler, '../models/feature_scaler.pkl')
print(f"✅ Best RF saved:  {best_name}")
print(f"✅ Scaler saved   ({'spatial 21-feature' if is_spatial else '7-feature'})")

metadata = {
    'model_type':        f'RandomForestClassifier ({best_name})',
    'n_features':        n_features,
    'feature_names':     feature_names,
    'classes':           [int(c) for c in np.unique(y)],
    'test_accuracy':     float(best_acc),
    'training_samples':  int(len(X_tr_sp if is_spatial else X_train)),
    'testing_samples':   int(len(X_te_sp if is_spatial else X_test))
}
with open('../models/model_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print("✅ Metadata saved to ../models/model_metadata.json")
print(f"\n📦 Best RF model ready — run 03b for CNN comparison")

✅ Best RF saved:  RF + Spatial Features
✅ Scaler saved   (spatial 21-feature)
✅ Metadata saved to ../models/model_metadata.json

📦 Best RF model ready — run 03b for CNN comparison
